In [0]:
import pandas as pd

In [0]:
df_calls = pd.read_csv("../../data/eda/cc_calls_eda.csv")

## Hypothesis 1: Contact ID vs Churn
- H0: Contact_ID is independent of churn.
- H1: Contact_ID is associated with churn.

In [0]:
import pandas as pd
from scipy.stats import chi2_contingency

data = df_calls[['Contact_ID', 'target']].dropna()

table = pd.crosstab(data['Contact_ID'], data['target'])

chi2, p, dof, exp = chi2_contingency(table)

print("P-value:", p)

if p > 0.05:
    print(" Fail to Reject H0 (as expected)")
else:
    print(" Unexpected result (check data leakage)")

## Hypothesis 2: External Consultant Mention vs Churn
- H0: External consultant mention is independent of churn.
- H1: External consultant mention affects churn.

In [0]:
data = df_calls[['cc_external_consultant', 'target']].dropna()

table = pd.crosstab(data['cc_external_consultant'], data['target'])

from scipy.stats import chi2_contingency

chi2, p, _, _ = chi2_contingency(table)

print("P-value:", p)

if p > 0.05:
    print("Fail to Reject H0 (likely)")
else:
    print("May have niche signal")

## Hypothesis 3: days_bucket vs Churn
- H0: days_bucket is not related to churn.
- H1: days_bucket has a meaningful effect on churn.

In [0]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

data = df_calls[['days_bucket', 'target']].dropna()

table = pd.crosstab(data['days_bucket'], data['target'])

chi2, p, dof, expected = chi2_contingency(table)

n = table.sum().sum()
r, k = table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

print(" Chi-Square Test: days_bucket vs Churn")
print("-" * 50)
print("Chi2:", chi2)
print("P-value:", p)
print("Cramér's V:", cramers_v)
print("\nContingency Table:\n", table)

alpha = 0.05

if p < alpha and cramers_v > 0.05:
    print("\n Reject H0: days_bucket has meaningful effect on churn")
elif p < alpha and cramers_v <= 0.05:
    print("\n Statistically significant but weak effect (likely not useful)")
else:
    print("\n Fail to Reject H0: days_bucket not related to churn")

print("\nExpected Frequencies:\n", expected)

In [0]:
print(df_calls['days_bucket'].value_counts())

## Hypothesis 4: Feature Screening vs Churn
- H0: The tested numeric and categorical features are independent of churn.
- H1: At least one tested feature is associated with churn.

In [0]:
import pandas as pd
from scipy.stats import mannwhitneyu, chi2_contingency

alpha = 0.05

fail_h0 = []
reject_h0 = []

num_cols = df_calls.select_dtypes(include=['int64','float64']).columns.drop('target', errors='ignore')
cat_cols = df_calls.select_dtypes(include=['object','category','bool']).columns

print("Finding features that fail to reject H0")
print("=" * 60)

for col in num_cols:
    data = df_calls[[col, 'target']].dropna()

    if data['target'].nunique() < 2:
        continue

    g1 = data[data['target'] == 1][col]
    g0 = data[data['target'] == 0][col]

    if len(g1) < 10 or len(g0) < 10:
        continue

    stat, p = mannwhitneyu(g1, g0, alternative='two-sided')

    print(f"[NUM] {col} -> p={p:.4f}")

    if p > alpha:
        fail_h0.append(col)
    else:
        reject_h0.append(col)

for col in cat_cols:
    data = df_calls[[col, 'target']].dropna()

    if data[col].nunique() < 2:
        continue

    table = pd.crosstab(data[col], data['target'])

    if table.shape[0] < 2:
        continue

    chi2, p, _, _ = chi2_contingency(table)

    print(f"[CAT] {col} -> p={p:.4f}")

    if p > alpha:
        fail_h0.append(col)
    else:
        reject_h0.append(col)

print("\n" + "=" * 60)
print("Features that fail to reject H0 (weak / non-drivers)")
print(fail_h0)

print("\nFeatures that reject H0 (strong drivers)")
print(reject_h0)

print("\nTotal fail H0:", len(fail_h0))
print("Total reject H0:", len(reject_h0))
print("=" * 60)

## Hypothesis 5: cc_contractor_suggest_leave vs Churn
- H0: cc_contractor_suggest_leave has no relationship with churn.
- H1: cc_contractor_suggest_leave is associated with churn.

In [0]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

data = df_calls[['cc_contractor_suggest_leave', 'target']].dropna()

table = pd.crosstab(data['cc_contractor_suggest_leave'], data['target'])

chi2, p, dof, expected = chi2_contingency(table)

n = table.sum().sum()
r, k = table.shape
cramers_v = np.sqrt(chi2 / (n * (min(r, k) - 1)))

print("Chi-Square Test: cc_contractor_suggest_leave vs Churn")
print("-" * 60)
print("Chi2:", chi2)
print("P-value:", p)
print("Cramér's V:", cramers_v)

print("\nContingency Table:\n", table)

alpha = 0.05

if p < alpha and cramers_v > 0.05:
    print("\nReject H0: Suggesting leave is strongly associated with churn")
elif p < alpha and cramers_v <= 0.05:
    print("\nStatistically significant but weak effect")
else:
    print("\nFail to Reject H0: No meaningful relationship with churn")

print("\nExpected Frequencies:\n", expected)

Due to class imbalance and sparsity, some tests may be unreliable in many cases.